In [ ]:
import torch
import numpy as np
from torchvision.datasets import DatasetFolder
from torch.utils.data import DataLoader, ConcatDataset, random_split


def doc_file_npy(duong_dan):
    mang_so = np.load(duong_dan)
    mang_so = (mang_so - np.min(mang_so)) / (np.max(mang_so) - np.min(mang_so) + 1e-8)
    anh_tensor = torch.tensor(mang_so, dtype=torch.float32)
    if anh_tensor.dim() == 2:
        anh_tensor = anh_tensor.unsqueeze(0)
    return anh_tensor

duong_dan_goc = 'dataset'


tap_train_goc = DatasetFolder(root=f'{duong_dan_goc}/train', loader=doc_file_npy, extensions=('.npy',))
tap_val_goc   = DatasetFolder(root=f'{duong_dan_goc}/val',   loader=doc_file_npy, extensions=('.npy',))

toan_bo_du_lieu =ConcatDataset([tap_train_goc, tap_val_goc])
tong_so_anh = len(toan_bo_du_lieu)

so_anh_train =int(0.9 * tong_so_anh)
so_anh_val = tong_so_anh-so_anh_train

tap_train_90, tap_val_10 =random_split(toan_bo_du_lieu, [so_anh_train, so_anh_val])

train_loader = DataLoader(tap_train_90, batch_size=32, shuffle=True)
val_loader = DataLoader(tap_val_10,   batch_size=32, shuffle=False)


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))


model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

model.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)


model.fc = nn.Linear(model.fc.in_features, 3)

model = model.to(device) 

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)


epochs = 10
print("ready")

for epoch in range(epochs):
    model.train() 
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        
        optimizer.zero_grad()               
        outputs = model(images)            
        loss = criterion(outputs, labels)   
        loss.backward()                     
        optimizer.step()                    
        
    
        running_loss += loss.item()
        
        
        if (i + 1) % 200 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], batch [{i+1}/{len(train_loader)}], loss: {running_loss/200:.4f}")
            running_loss = 0.0 



In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt


model.eval() 

danh_sach_dap_an_that = []
danh_sach_du_doan = []

# Tắt bộ tính đạo hàm (tiết kiệm RAM và tăng tốc)
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        
        # Đưa ảnh vào dự đoán
        outputs = model(images)
        # Biến đổi điểm số thô thành xác suất phần trăm (0 đến 1)
        probs = torch.softmax(outputs, dim=1) 
        
        # Gom kết quả lại
        danh_sach_dap_an_that.extend(labels.cpu().numpy())
        danh_sach_du_doan.extend(probs.cpu().numpy())

danh_sach_dap_an_that =np.array(danh_sach_dap_an_that)
danh_sach_du_doan =np.array(danh_sach_du_doan)

# --- VẼ ĐƯỜNG CONG ROC ---
plt.figure(figsize=(10, 8))

# Lấy tên các nhãn (no, sphere, vort) từ tập gốc
classes = toan_bo_du_lieu.datasets[0].classes 

for i in range(3):
    # Kỹ thuật One-vs-Rest: Tách từng class ra để vẽ
    binary_labels =(danh_sach_dap_an_that == i).astype(int)
    fpr, tpr, _ =roc_curve(binary_labels, danh_sach_du_doan[:, i])
    
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, lw=2, label=f'Class {classes[i]} (AUC = {roc_auc:.4f})')

# Căn chỉnh đồ thị cho đẹp mắt
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate ')
plt.ylabel('True Positive Rate')
plt.title(' ROC ')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()